# 01. First QTT Function and Grid

## Learning goals

- see how a quantics grid turns one interval into `2^R` sample points
- build a QTT approximation of `cosh(x)` with the public Julia API
- inspect the bond dimensions of the resulting tensor train
- compare the QTT values with the exact function values
- repeat the workflow once more on a shifted interval with a second function

In [ ]:
using Tensor4all
using CairoMakie

const QG = Tensor4all.QuanticsGrids
const QTCI = Tensor4all.QuanticsTCI
const TN = Tensor4all.TensorNetworks
const STT = Tensor4all.SimpleTT

## Quantics grid in one dimension

A one-dimensional quantics grid is controlled by a bit depth `R`.
That means `2^R` sample points on the chosen interval.

Here we use `DiscretizedGrid{1}` with `includeendpoint=true` so the grid covers the closed interval `[0, 1]`.
The helper `grididx_to_origcoord` maps a 1-based grid index back to the physical coordinate, and `grididx_to_quantics` shows the binary site values used by the QTT representation.

In [ ]:
R = 7
npoints = 1 << R
grid = QG.DiscretizedGrid{1}(R, 0.0, 1.0; includeendpoint=true)
xvals = [QG.grididx_to_origcoord(grid, i) for i in 1:npoints]

@show R npoints
@show first(xvals) last(xvals)
@show QG.grididx_to_quantics(grid, 1)
@show QG.grididx_to_quantics(grid, npoints)

## Main walkthrough: `cosh(x)`

The first function is `cosh(x)`, because it is smooth and easy to reason about.
The QTT construction uses the public `QuanticsTCI` wrapper from `Tensor4all.jl`.

After the interpolation step, we convert the result into the raw tensor-train layer with `SimpleTT.TensorTrain`, then into the indexed chain layer with `TensorNetworks.TensorTrain` so we can inspect the bond dimensions and evaluate a sample through the tensor-network API.

In [ ]:
target_function(x) = cosh(x)

qtt, ranks, errors = QTCI.quanticscrossinterpolate(
    Float64,
    target_function,
    grid;
    tolerance=1e-12,
    maxbonddim=32,
    nrandominitpivot=3,
)

simple_tt = STT.TensorTrain(qtt.tci)
sites = [Tensor4all.Index(2; tags=["x", "bit=$i"]) for i in 1:length(simple_tt)]
indexed_tt = TN.TensorTrain(simple_tt, sites)
bond_dims = TN.linkdims(indexed_tt)

@show length(simple_tt)
@show bond_dims
@show last(ranks)
@show last(errors)

In [ ]:
cosh_exact = target_function.(xvals)
cosh_qtt = [real(qtt(i)) for i in 1:npoints]
cosh_max_abs_error = maximum(abs.(cosh_exact .- cosh_qtt))

sample_i = 17
sample_digits = QG.grididx_to_quantics(grid, sample_i)
sample_qtt = qtt(sample_i)
sample_tn = real(TN.evaluate(indexed_tt, sites, sample_digits))
sample_exact = target_function(QG.grididx_to_origcoord(grid, sample_i))

@show sample_i sample_digits
@show sample_qtt sample_tn sample_exact
@show cosh_max_abs_error

In [ ]:
fig = Figure(size=(1000, 380))

ax1 = Axis(
    fig[1, 1],
    xlabel="x",
    ylabel="value",
    title="cosh(x) on a quantics grid",
)
lines!(ax1, xvals, cosh_exact; color=:black, linewidth=2, label="exact cosh(x)")
scatter!(ax1, xvals, cosh_qtt; color=:dodgerblue, markersize=5, label="QTT samples")
axislegend(ax1; position=:lt)

ax2 = Axis(
    fig[1, 2],
    xlabel="bond link",
    ylabel="bond dimension",
    title="Internal bond dimensions",
)
lines!(ax2, 1:length(bond_dims), bond_dims; color=:darkorange, linewidth=2)

fig

## Why `cosh(x)` stays compact

The function `cosh(x)` is a good first example because

    cosh(x) = (exp(x) + exp(-x)) / 2.

On a quantics grid, the binary digits of the grid index contribute additively to the coordinate `x`. Exponentials turn sums into products, so `exp(x)` and `exp(-x)` can be represented very compactly across the bit sites. Since `cosh(x)` is a sum of two such exponentials, seeing bond dimensions close to `2` is not accidental: it reflects the structure of the function.


## Second experiment: a shifted interval

The same workflow also works on a shifted interval.
Here the grid covers `[-1, 2]`, and the second function has a bit more structure so the approximation is less trivial.
Try changing `experiment_function`, the interval bounds, or `R`, then rerun the next two cells and compare the new error and bond dimensions.


In [ ]:
experiment_function(x) = exp(-4 * (x - 0.25)^2) * cos(5x)

experiment_grid = QG.DiscretizedGrid{1}(R, -1.0, 2.0; includeendpoint=true)
experiment_xvals = [QG.grididx_to_origcoord(experiment_grid, i) for i in 1:npoints]

experiment_qtt, experiment_ranks, experiment_errors = QTCI.quanticscrossinterpolate(
    Float64,
    experiment_function,
    experiment_grid;
    tolerance=1e-11,
    maxbonddim=48,
    nrandominitpivot=3,
)

experiment_simple_tt = STT.TensorTrain(experiment_qtt.tci)
experiment_sites = [Tensor4all.Index(2; tags=["u", "bit=$i"]) for i in 1:length(experiment_simple_tt)]
experiment_indexed_tt = TN.TensorTrain(experiment_simple_tt, experiment_sites)
experiment_bond_dims = TN.linkdims(experiment_indexed_tt)

experiment_exact = experiment_function.(experiment_xvals)
experiment_values = [real(experiment_qtt(i)) for i in 1:npoints]
experiment_max_abs_error = maximum(abs.(experiment_exact .- experiment_values))

@show first(experiment_xvals) last(experiment_xvals)
@show experiment_bond_dims
@show last(experiment_ranks) last(experiment_errors)
@show experiment_max_abs_error

In [ ]:
fig2 = Figure(size=(1000, 380))

ax3 = Axis(
    fig2[1, 1],
    xlabel="x",
    ylabel="value",
    title="Second experiment on [-1, 2]",
)
lines!(ax3, experiment_xvals, experiment_exact; color=:black, linewidth=2, label="exact function")
scatter!(ax3, experiment_xvals, experiment_values; color=:seagreen, markersize=5, label="QTT samples")
axislegend(ax3; position=:lt)

ax4 = Axis(
    fig2[1, 2],
    xlabel="bond link",
    ylabel="bond dimension",
    title="Second experiment bond dimensions",
)
lines!(ax4, 1:length(experiment_bond_dims), experiment_bond_dims; color=:firebrick, linewidth=2)

fig2

## What to notice

- `R` controls the bit depth and therefore the number of grid points.
- `QuanticsTCI.quanticscrossinterpolate` builds the QTT from a function callback on a quantics grid.
- `SimpleTT.TensorTrain(qtt.tci)` exposes the raw tensor cores.
- `TensorNetworks.TensorTrain(simple_tt, sites)` turns the raw TT into an indexed chain.
- `TensorNetworks.evaluate` reads a value back from the indexed chain.
- `TensorNetworks.linkdims` shows the internal bond-dimension profile.

## API recap

- `Tensor4all.QuanticsGrids.DiscretizedGrid{1}`
- `Tensor4all.QuanticsGrids.grididx_to_origcoord`
- `Tensor4all.QuanticsGrids.grididx_to_quantics`
- `Tensor4all.QuanticsTCI.quanticscrossinterpolate`
- `Tensor4all.SimpleTT.TensorTrain`
- `Tensor4all.TensorNetworks.TensorTrain`
- `Tensor4all.TensorNetworks.evaluate`
- `Tensor4all.TensorNetworks.linkdims`

Notebook 02 will pick up the accuracy and bond-dimension story in more detail.